In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

plt.close('all')

ROOT = Path.cwd().parent
processed = ROOT / "data" / "processed"

# ── Load model and test artifacts ──────────────────────────────────────────────
xgb_model     = joblib.load(ROOT / "models" / "fraud_detection_model.pkl")
X_test        = joblib.load(processed / "X_test.pkl")
y_test        = joblib.load(processed / "y_test.pkl")
X_train_smote = joblib.load(processed / "X_train_smote.pkl")
print("Files loaded successfully")

# ── Fraud probability scores ───────────────────────────────────────────────────
# Always use predict_proba()[:,1] for threshold-independent metrics (ROC-AUC, PR-AUC).
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# ── ROC Curve ──────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — XGBoost")
plt.legend()
plt.tight_layout()
plt.show()

# ── Precision-Recall Curve ─────────────────────────────────────────────────────
# PR-AUC is the primary metric for imbalanced fraud detection:
# it shows how precision degrades as recall increases, without being
# inflated by the dominant negative class.
precision_arr, recall_arr, thresholds = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recall_arr, precision_arr, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — XGBoost")
plt.legend()
plt.tight_layout()
plt.show()

# ── Threshold analysis ─────────────────────────────────────────────────────────
# Default threshold of 0.10 is used in production (src/predict.py) to favour recall.
# Explore the trade-off here to justify or adjust that choice.
for threshold in [0.05, 0.10, 0.20, 0.30, 0.50]:
    y_pred_t = (y_prob >= threshold).astype(int)
    from sklearn.metrics import precision_score, recall_score, f1_score
    p = precision_score(y_test, y_pred_t)
    r = recall_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print(f"Threshold {threshold:.2f}  |  Precision: {p:.3f}  Recall: {r:.3f}  F1: {f:.3f}")

# Show confusion matrix at the chosen production threshold.
THRESHOLD = 0.10
y_pred = (y_prob >= THRESHOLD).astype(int)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
plt.title(f'Confusion Matrix at Threshold = {THRESHOLD}')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print(f"\nClassification report at threshold = {THRESHOLD}:\n")
print(classification_report(y_test, y_pred))

# ── SHAP Explainability ────────────────────────────────────────────────────────
# TreeExplainer decomposes each prediction into per-feature contributions,
# showing WHICH features drove the model's decision on each transaction.
explainer = shap.TreeExplainer(xgb_model)
sample_data = X_test.sample(1000, random_state=42)
shap_values = explainer.shap_values(sample_data)
print("SHAP values computed on 1,000-row sample")

shap.summary_plot(shap_values, sample_data, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (mean |SHAP value|)")
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, sample_data, show=False)
plt.title("SHAP Summary — impact of each feature on fraud score")
plt.tight_layout()
plt.show()

# ── Feature Importance (model's built-in) ─────────────────────────────────────
importance_df = pd.DataFrame({
    'Feature':    X_train_smote.columns,
    'Importance': xgb_model.feature_importances_,
}).sort_values('Importance', ascending=False)

print("\nTop 15 features by XGBoost gain:\n", importance_df.head(15).to_string(index=False))

plt.figure(figsize=(10, 7))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
plt.title("Top 15 Feature Importance — XGBoost")
plt.tight_layout()
plt.show()

# ── Save feature importance ────────────────────────────────────────────────────
reports_dir = ROOT / "reports"
reports_dir.mkdir(exist_ok=True)
importance_df.to_csv(reports_dir / "feature_importance.csv", index=False)
print("Feature importance saved to reports/feature_importance.csv")

# ── Fraud probability distribution ────────────────────────────────────────────
plt.figure(figsize=(8, 5))
sns.histplot(y_prob[y_test == 0], bins=50, label='Legitimate', alpha=0.7)
sns.histplot(y_prob[y_test == 1], bins=50, label='Fraud', alpha=0.7, color='red')
plt.xlabel("Predicted Fraud Probability")
plt.title("Probability Distribution by True Class")
plt.legend()
plt.tight_layout()
plt.show()

print("\nModel evaluation complete.")